# Discovery + Silver: `university.enrollments`

Tabla puente: un estudiante inscrito a un curso en un semestre. Tres FKs a validar (`student_id`, `course_id`, `semester_id`).

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.university__enrollments", engine)
df.shape

(25000, 9)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

enrollment_id            object
enrolled_at              object
status                   object
student_id               object
course_id                object
semester_id              object
_source_file             object
_ingested_at     datetime64[ns]
_dag_run_id              object
dtype: object


,enrollment_id,enrolled_at,status,student_id,course_id,semester_id,_source_file,_ingested_at,_dag_run_id
0,ENR-00000001,2022-03-25,completed,STU-0003502,CRS-00096,SEM-004,university/enrollments.csv,2026-07-15 22:29:48.053653,manual__2026-07-15T22:29:45+00:00
1,ENR-00000002,2025-03-05,completed,STU-0002979,CRS-00205,SEM-003,university/enrollments.csv,2026-07-15 22:29:48.053653,manual__2026-07-15T22:29:45+00:00
2,ENR-00000003,2024-07-12,completed,STU-0003489,CRS-00056,SEM-002,university/enrollments.csv,2026-07-15 22:29:48.053653,manual__2026-07-15T22:29:45+00:00
3,ENR-00000004,2023-06-21,completed,STU-0003134,CRS-00112,SEM-004,university/enrollments.csv,2026-07-15 22:29:48.053653,manual__2026-07-15T22:29:45+00:00
4,ENR-00000005,2023-07-24,dropped,STU-0001017,CRS-00284,SEM-006,university/enrollments.csv,2026-07-15 22:29:48.053653,manual__2026-07-15T22:29:45+00:00


## 2. Nulos, duplicados e integridad referencial

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("enrollment_id duplicados:", df["enrollment_id"].duplicated().sum())

students = pd.read_sql("SELECT student_id FROM silver.university__students", engine)
courses = pd.read_sql("SELECT course_id FROM silver.university__courses", engine)
semesters = pd.read_sql("SELECT semester_id FROM silver.university__semesters", engine)

print("student_id huerfanos:", (~df["student_id"].isin(students["student_id"])).sum())
print("course_id huerfanos:", (~df["course_id"].isin(courses["course_id"])).sum())
print("semester_id huerfanos:", (~df["semester_id"].isin(semesters["semester_id"])).sum())

Nulos por columna:
enrollment_id    0
enrolled_at      0
status           0
student_id       0
course_id        0
semester_id      0
_source_file     0
_ingested_at     0
_dag_run_id      0
dtype: int64

enrollment_id duplicados: 0
student_id huerfanos: 0
course_id huerfanos: 0
semester_id huerfanos: 0


## 3. `status`: valores y `enrolled_at`

In [4]:
print("Valores distintos de status:")
print(df["status"].value_counts())
print()
enrolled = pd.to_datetime(df["enrolled_at"])
print("enrolled_at invalidas:", enrolled.isna().sum())

Valores distintos de status:
status
completed    14931
active        5035
failed        2531
dropped       2503
Name: count, dtype: int64

enrolled_at invalidas: 0


## 4. Conclusion

Tabla limpia (sin nulos, sin duplicados, 0 FKs huerfanas, `status` con 4 valores consistentes: active/completed/dropped/failed). Solo tipado: `enrolled_at` a fecha, `status` estandarizado.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["enrollment_id", "student_id", "course_id", "semester_id", "status", "enrolled_at"]].copy()

df_silver["status"] = df_silver["status"].str.strip().str.lower()
df_silver["enrolled_at"] = pd.to_datetime(df_silver["enrolled_at"]).dt.date

df_silver.head()

,enrollment_id,student_id,course_id,semester_id,status,enrolled_at
0,ENR-00000001,STU-0003502,CRS-00096,SEM-004,completed,2022-03-25
1,ENR-00000002,STU-0002979,CRS-00205,SEM-003,completed,2025-03-05
2,ENR-00000003,STU-0003489,CRS-00056,SEM-002,completed,2024-07-12
3,ENR-00000004,STU-0003134,CRS-00112,SEM-004,completed,2023-06-21
4,ENR-00000005,STU-0001017,CRS-00284,SEM-006,dropped,2023-07-24


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver.isna().sum().sum() == 0
assert df_silver["enrollment_id"].is_unique
assert df_silver["student_id"].isin(students["student_id"]).all()
assert df_silver["course_id"].isin(courses["course_id"]).all()
assert df_silver["semester_id"].isin(semesters["semester_id"]).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 25000 filas listas para silver


## 7. Escribir en `silver.university__enrollments`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

df_silver.to_sql(
    "university__enrollments",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=2000,
)
print("Escrito en silver.university__enrollments")

Escrito en silver.university__enrollments


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.university__enrollments LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT enrollment_id) AS ids_unicos FROM silver.university__enrollments", engine))
check

   filas  ids_unicos
0  25000       25000


,enrollment_id,student_id,course_id,semester_id,status,enrolled_at,_silver_loaded_at
0,ENR-00000001,STU-0003502,CRS-00096,SEM-004,completed,2022-03-25,2026-07-16 19:03:30.708609+00:00
1,ENR-00000002,STU-0002979,CRS-00205,SEM-003,completed,2025-03-05,2026-07-16 19:03:30.708609+00:00
2,ENR-00000003,STU-0003489,CRS-00056,SEM-002,completed,2024-07-12,2026-07-16 19:03:30.708609+00:00
3,ENR-00000004,STU-0003134,CRS-00112,SEM-004,completed,2023-06-21,2026-07-16 19:03:30.708609+00:00
4,ENR-00000005,STU-0001017,CRS-00284,SEM-006,dropped,2023-07-24,2026-07-16 19:03:30.708609+00:00
